<a href="https://colab.research.google.com/github/AngeloSorte/Banking-Customer-Risk-Simulation-with-SQL-and-ML/blob/angelosorte.github.io/Banking_Customer_Risk_Simulation_with_SQL_and_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ====== IMPORT ======
import sqlite3
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ====== REMOVE EXISTING DATABASE ======
if os.path.exists("bank.db"):
    os.remove("bank.db")

# ====== CREATE DATABASE ======
conn = sqlite3.connect("bank.db", timeout=10)
cursor = conn.cursor()

# ====== CREATE TABLES ======
cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    age INTEGER,
    income REAL
)
""")

cursor.execute("""
CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    amount REAL,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
)
""")
conn.commit()

# ====== GENERATE DATA ======
np.random.seed(42)

# Insert customers
customers = [(i, np.random.randint(18, 70), np.random.randint(20000, 100000)) for i in range(1, 501)]
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)

# Insert transactions
transactions = []
for i in range(1, 2001):
    customer_id = np.random.randint(1, 501)
    amount = np.random.randint(10, 2000)
    transactions.append((i, customer_id, amount))

cursor.executemany("INSERT INTO transactions VALUES (?, ?, ?)", transactions)
conn.commit()

# ====== SQL QUERY (FEATURE ENGINEERING) ======
query = """
SELECT
    c.customer_id,
    c.age,
    c.income,
    COUNT(t.transaction_id) as num_transactions,
    AVG(t.amount) as avg_amount,
    SUM(t.amount) as total_amount
FROM customers c
LEFT JOIN transactions t
ON c.customer_id = t.customer_id
GROUP BY c.customer_id
"""

df = pd.read_sql_query(query, conn)

# ====== HANDLE MISSING VALUES ======
df['num_transactions'] = df['num_transactions'].fillna(0)
df['avg_amount'] = df['avg_amount'].fillna(0)
df['total_amount'] = df['total_amount'].fillna(0)

# ====== CREATE TARGET (SIMULATED RISK, BALANCED) ======
# Initial rule-based risk
df['risk'] = df.apply(lambda row: 1 if (row['income'] < 40000 and row['total_amount'] > 15000) else 0, axis=1)

# Ensure at least 30% high risk
num_high_risk = df['risk'].sum()
target_high_risk = int(0.3 * len(df))
if num_high_risk < target_high_risk:
    missing = target_high_risk - num_high_risk
    zero_indices = df[df['risk'] == 0].index
    add_indices = np.random.choice(zero_indices, missing, replace=False)
    df.loc[add_indices, 'risk'] = 1

# ====== FEATURES ======
X = df[['age', 'income', 'num_transactions', 'avg_amount', 'total_amount']]
y = df['risk']

# ====== TRAIN TEST SPLIT ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ====== SCALING ======
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====== MODEL ======
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ====== EVALUATION ======
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# ====== SAMPLE PREDICTION ======
sample = [[35, 30000, 10, 500, 25000]]  # Example customer
sample_scaled = scaler.transform(sample)
prediction = model.predict(sample_scaled)
print("\nSample prediction:", "High Risk" if prediction[0] == 1 else "Low Risk")

# ====== CLOSE CONNECTION ======
conn.close()

Accuracy: 0.744
              precision    recall  f1-score   support

           0       0.74      1.00      0.85        93
           1       0.00      0.00      0.00        32

    accuracy                           0.74       125
   macro avg       0.37      0.50      0.43       125
weighted avg       0.55      0.74      0.63       125


Sample prediction: Low Risk


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/u